In [2]:
import pandas as pd
import numpy as np

# Load student-level split assignments
split_assignments = pd.read_csv(
    "../data/processed/split_assignments.csv"
)

# Retain only training-pool students
train_assignments = split_assignments[
    split_assignments["partition"] == "train_pool"
].copy()

# Convert validation-fold numbers to integers
train_assignments["validation_fold"] = (
    train_assignments["validation_fold"].astype(int)
)

print("Training-pool students:", train_assignments["user_id"].nunique())

print("\nValidation fold distribution:")
print(
    train_assignments["validation_fold"]
    .value_counts()
    .sort_index()
)

Training-pool students: 3373

Validation fold distribution:
validation_fold
1    675
2    675
3    675
4    674
5    674
Name: count, dtype: int64


In [3]:
# Load cleaned interaction records
clean_df = pd.read_csv(
    "../data/processed/main_questions_cleaned.csv"
)

# Retain interactions belonging to training-pool students
train_df = clean_df[
    clean_df["user_id"].isin(train_assignments["user_id"])
].copy()

print("Training interaction shape:", train_df.shape)
print("Unique training students:", train_df["user_id"].nunique())

Training interaction shape: (219983, 7)
Unique training students: 3373


In [4]:
fold_datasets = []

for fold_number in range(1, 6):

    # Identify validation students for this round
    val_student_ids = train_assignments.loc[
        train_assignments["validation_fold"] == fold_number,
        "user_id"
    ]

    # All remaining training-pool students are training students
    fold_train_student_ids = train_assignments.loc[
        train_assignments["validation_fold"] != fold_number,
        "user_id"
    ]

    # Extract their interaction records
    fold_train_df = train_df[
        train_df["user_id"].isin(fold_train_student_ids)
    ].copy()

    fold_val_df = train_df[
        train_df["user_id"].isin(val_student_ids)
    ].copy()

    # Store this round's training and validation datasets
    fold_datasets.append((fold_train_df, fold_val_df))

    print(
        f"Fold {fold_number}: "
        f"Training rows = {len(fold_train_df)}, "
        f"Validation rows = {len(fold_val_df)}"
    )

Fold 1: Training rows = 177324, Validation rows = 42659
Fold 2: Training rows = 176273, Validation rows = 43710
Fold 3: Training rows = 170186, Validation rows = 49797
Fold 4: Training rows = 178311, Validation rows = 41672
Fold 5: Training rows = 177838, Validation rows = 42145


In [5]:
for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    train_users = set(fold_train_df["user_id"].unique())
    val_users = set(fold_val_df["user_id"].unique())

    print(
        f"Fold {fold_number}: "
        f"Overlap = {len(train_users & val_users)}, "
        f"Training students = {len(train_users)}, "
        f"Validation students = {len(val_users)}"
    )

Fold 1: Overlap = 0, Training students = 2698, Validation students = 675
Fold 2: Overlap = 0, Training students = 2698, Validation students = 675
Fold 3: Overlap = 0, Training students = 2698, Validation students = 675
Fold 4: Overlap = 0, Training students = 2699, Validation students = 674
Fold 5: Overlap = 0, Training students = 2699, Validation students = 674


In [6]:
baseline_results = []

for fold_number, (fold_train_df, fold_val_df) in enumerate(
    fold_datasets, start=1
):

    # Count correct outcomes in the training data
    correct_count = fold_train_df["correct"].sum()

    # Total number of training interactions
    total_count = len(fold_train_df)

    # Calculate training correct rate
    baseline_probability = correct_count / total_count

    baseline_results.append({
        "fold": fold_number,
        "correct_count": correct_count,
        "total_count": total_count,
        "baseline_probability": baseline_probability
    })

    print(
        f"Fold {fold_number}: "
        f"Correct = {correct_count}, "
        f"Total = {total_count}, "
        f"Baseline probability = {baseline_probability:.4f}"
    )

Fold 1: Correct = 117058, Total = 177324, Baseline probability = 0.6601
Fold 2: Correct = 115370, Total = 176273, Baseline probability = 0.6545
Fold 3: Correct = 112926, Total = 170186, Baseline probability = 0.6635
Fold 4: Correct = 117253, Total = 178311, Baseline probability = 0.6576
Fold 5: Correct = 116937, Total = 177838, Baseline probability = 0.6575


In [7]:
from sklearn.metrics import log_loss, brier_score_loss
import numpy as np

fold_metrics = []

for fold_number, ((fold_train_df, fold_val_df), result) in enumerate(
    zip(fold_datasets, baseline_results), start=1
):

    # Actual outcomes in the validation set
    y_val = fold_val_df["correct"].to_numpy()

    # Predict the same training probability for every validation row
    y_pred = np.full(
        len(fold_val_df),
        result["baseline_probability"]
    )

    # Calculate evaluation metrics
    ll = log_loss(y_val, y_pred, labels=[0, 1])
    bs = brier_score_loss(y_val, y_pred)

    fold_metrics.append({
        "fold": fold_number,
        "log_loss": ll,
        "brier_score": bs
    })

    print(
        f"Fold {fold_number}: "
        f"Log Loss = {ll:.4f}, "
        f"Brier Score = {bs:.4f}"
    )

Fold 1: Log Loss = 0.6461, Brier Score = 0.2269
Fold 2: Log Loss = 0.6313, Brier Score = 0.2197
Fold 3: Log Loss = 0.6534, Brier Score = 0.2304
Fold 4: Log Loss = 0.6390, Brier Score = 0.2234
Fold 5: Log Loss = 0.6390, Brier Score = 0.2234


In [8]:
avg_log_loss = np.mean([
    result["log_loss"] for result in fold_metrics
])

avg_brier_score = np.mean([
    result["brier_score"] for result in fold_metrics
])

print(f"V0 Average Log Loss: {avg_log_loss:.4f}")
print(f"V0 Average Brier Score: {avg_brier_score:.4f}")

V0 Average Log Loss: 0.6418
V0 Average Brier Score: 0.2248
